# Paper scope audit and MediaPipe calibration validation

This notebook does two things:

1. **Scope audit for paper claims** ? records which claims are supported by the current data and which should be removed or marked as future work.
2. **Held-out calibration validation** ? checks whether the MediaPipe-to-OptiTrack affine alignment generalizes when entire target radii or directions are held out.

This supports a conservative paper statement based on the actual available experiment:

- natural movement, not explicitly slow vs fast
- no tested hand orientation set N/NE/NW
- no tested clockwise/counterclockwise comparison
- without finger contact only, according to the current experiment description

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path('.')
COMPARE_DIR = ROOT / 'OT-results' / 'tracker_comparison'
PER_TARGET_DIR = COMPARE_DIR / 'per_target_error'
OUT_DIR = COMPARE_DIR / 'paper_scope_validation'
PLOT_DIR = OUT_DIR / 'plots'
for d in [OUT_DIR, PLOT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

LABELED_CSV = PER_TARGET_DIR / 'all_labeled_error_samples.csv'
SEGMENT_METRICS_CSV = COMPARE_DIR / 'segment_error_metrics.csv'
FINGER_METRICS_CSV = COMPARE_DIR / 'finger_error_metrics.csv'

MIN_TRAIN_SAMPLES = 200
MIN_TEST_SAMPLES = 30

plt.rcParams['figure.dpi'] = 130
plt.rcParams['savefig.dpi'] = 180
plt.rcParams['axes.grid'] = True

In [ ]:
def fit_affine(src_xy, dst_xz):
    X = np.column_stack([src_xy[:, 0], src_xy[:, 1], np.ones(len(src_xy))])
    B, *_ = np.linalg.lstsq(X, dst_xz, rcond=None)
    return B


def apply_affine(src_xy, B):
    X = np.column_stack([src_xy[:, 0], src_xy[:, 1], np.ones(len(src_xy))])
    return X @ B


def metrics(errors):
    e = np.asarray(errors, dtype=float)
    e = e[np.isfinite(e)]
    if len(e) == 0:
        return {'n': 0, 'rmse_mm': np.nan, 'mean_error_mm': np.nan, 'p95_error_mm': np.nan, 'median_error_mm': np.nan}
    return {
        'n': int(len(e)),
        'rmse_mm': float(np.sqrt(np.mean(e ** 2))),
        'mean_error_mm': float(np.mean(e)),
        'p95_error_mm': float(np.percentile(e, 95)),
        'median_error_mm': float(np.median(e)),
    }


def evaluate_holdout(df, holdout_col, include_center=True):
    rows = []
    for (finger, mp_pair, ot_file), seg in df.groupby(['finger', 'mp_pair', 'ot_file']):
        seg = seg.copy()
        if not include_center:
            seg = seg[seg['target_direction'] != 'CENTER'].copy()
        labels = sorted(seg[holdout_col].dropna().unique())
        for label in labels:
            test = seg[holdout_col].eq(label)
            train = ~test
            if train.sum() < MIN_TRAIN_SAMPLES or test.sum() < MIN_TEST_SAMPLES:
                continue
            src_train = seg.loc[train, ['mp_x_px_smoothed', 'mp_y_px_smoothed']].to_numpy(dtype=float)
            dst_train = seg.loc[train, ['ot_X_mm', 'ot_Z_mm']].to_numpy(dtype=float)
            src_test = seg.loc[test, ['mp_x_px_smoothed', 'mp_y_px_smoothed']].to_numpy(dtype=float)
            dst_test = seg.loc[test, ['ot_X_mm', 'ot_Z_mm']].to_numpy(dtype=float)
            B = fit_affine(src_train, dst_train)
            pred = apply_affine(src_test, B)
            err = np.linalg.norm(pred - dst_test, axis=1)
            m = metrics(err)
            rows.append({
                'finger': finger,
                'mp_pair': mp_pair,
                'ot_file': ot_file,
                'holdout_type': holdout_col,
                'held_out': label,
                'train_samples': int(train.sum()),
                **m,
            })
    return pd.DataFrame(rows)


def aggregate_holdout(df):
    rows=[]
    for keys, g in df.groupby(['holdout_type','held_out']):
        n = g['n'].sum()
        rmse = np.sqrt(np.sum((g['rmse_mm'] ** 2) * g['n']) / n) if n else np.nan
        mean = np.sum(g['mean_error_mm'] * g['n']) / n if n else np.nan
        p95_weighted = np.sum(g['p95_error_mm'] * g['n']) / n if n else np.nan
        rows.append({'holdout_type': keys[0], 'held_out': keys[1], 'segments': len(g), 'n': int(n), 'rmse_mm': rmse, 'mean_error_mm': mean, 'weighted_p95_error_mm': p95_weighted})
    return pd.DataFrame(rows)

In [ ]:
labeled = pd.read_csv(LABELED_CSV)
segment_metrics = pd.read_csv(SEGMENT_METRICS_CSV)
finger_metrics = pd.read_csv(FINGER_METRICS_CSV)

labeled = labeled[np.isfinite(labeled['ot_X_mm']) & np.isfinite(labeled['ot_Z_mm'])].copy()

radius_holdout = evaluate_holdout(labeled, 'target_radius_cm', include_center=True)
direction_holdout = evaluate_holdout(labeled[labeled['target_direction'] != 'CENTER'], 'target_direction', include_center=False)
holdout_all = pd.concat([radius_holdout, direction_holdout], ignore_index=True)
holdout_summary = aggregate_holdout(holdout_all)

radius_holdout.to_csv(OUT_DIR / 'leave_one_radius_out_calibration_validation.csv', index=False)
direction_holdout.to_csv(OUT_DIR / 'leave_one_direction_out_calibration_validation.csv', index=False)
holdout_all.to_csv(OUT_DIR / 'heldout_calibration_validation_by_segment.csv', index=False)
holdout_summary.to_csv(OUT_DIR / 'heldout_calibration_validation_summary.csv', index=False)

print('Current calibrated per-finger metrics from main comparison:')
print(finger_metrics[['finger','segments_compared','samples_compared','rmse_mm','mean_positional_error_mm','p95_positional_error_mm']].to_string(index=False, float_format=lambda x: f'{x:.3f}'))
print('\nLeave-one-radius-out summary:')
print(holdout_summary[holdout_summary['holdout_type'].eq('target_radius_cm')].to_string(index=False, float_format=lambda x: f'{x:.3f}'))
print('\nLeave-one-direction-out summary:')
print(holdout_summary[holdout_summary['holdout_type'].eq('target_direction')].to_string(index=False, float_format=lambda x: f'{x:.3f}'))

In [ ]:
rad = holdout_summary[holdout_summary['holdout_type'].eq('target_radius_cm')].copy()
rad['held_out_num'] = pd.to_numeric(rad['held_out'])
rad = rad.sort_values('held_out_num')
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(rad['held_out_num'], rad['rmse_mm'], marker='o', label='RMSE')
ax.plot(rad['held_out_num'], rad['mean_error_mm'], marker='o', label='Mean')
ax.plot(rad['held_out_num'], rad['weighted_p95_error_mm'], marker='o', label='Weighted p95')
ax.set_xlabel('Held-out target radius (cm)')
ax.set_ylabel('Held-out positional error (mm)')
ax.set_title('Calibration validation: leave-one-radius-out')
ax.legend()
fig.tight_layout()
fig.savefig(PLOT_DIR / 'leave_one_radius_out_validation.png', bbox_inches='tight')
plt.close(fig)

dir_order = ['E','NE','N','NW','W','SW','S','SE']
dir_df = holdout_summary[holdout_summary['holdout_type'].eq('target_direction')].copy()
dir_df['order'] = dir_df['held_out'].map({d:i for i,d in enumerate(dir_order)})
dir_df = dir_df.sort_values('order')
fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(dir_df['held_out'], dir_df['rmse_mm'], marker='o', label='RMSE')
ax.plot(dir_df['held_out'], dir_df['mean_error_mm'], marker='o', label='Mean')
ax.plot(dir_df['held_out'], dir_df['weighted_p95_error_mm'], marker='o', label='Weighted p95')
ax.set_xlabel('Held-out target direction')
ax.set_ylabel('Held-out positional error (mm)')
ax.set_title('Calibration validation: leave-one-direction-out')
ax.legend()
fig.tight_layout()
fig.savefig(PLOT_DIR / 'leave_one_direction_out_validation.png', bbox_inches='tight')
plt.close(fig)

print(PLOT_DIR / 'leave_one_radius_out_validation.png')
print(PLOT_DIR / 'leave_one_direction_out_validation.png')

In [ ]:
coverage_rows = [
    {'paper_claim_or_factor': 'Four fingers', 'supported_by_current_data': 'yes', 'evidence': 'Mediapipe pairs: index, middle, ring, pinky; 2 matched OptiTrack segments per finger after clock correction.', 'recommendation': 'Keep, but report n=2 matched segments per finger for this dataset.'},
    {'paper_claim_or_factor': 'Targets at 0, 1, 2, 3, 5, 8 cm on eight directions', 'supported_by_current_data': 'yes, for target labeling', 'evidence': 'object_x/object_y trajectory spans the target grid; per-target tables generated.', 'recommendation': 'Keep if this was the commanded protocol. Report per-target error table/heatmap.'},
    {'paper_claim_or_factor': 'Three hand orientations: N, NE, NW', 'supported_by_current_data': 'no', 'evidence': 'No orientation labels or repeated orientation folders found in the provided dataset.', 'recommendation': 'Remove from methods/results, or collect orientation data before claiming orientation robustness.'},
    {'paper_claim_or_factor': 'Clockwise and counterclockwise traversal', 'supported_by_current_data': 'no / not separable', 'evidence': 'Current logs do not label direction order; natural trajectory appears continuous but not condition-coded CW/CCW.', 'recommendation': 'Do not claim CW/CCW comparison unless you add explicit labels or trials.'},
    {'paper_claim_or_factor': 'Five slow and five fast repetitions', 'supported_by_current_data': 'no', 'evidence': 'User notes movement was natural, not controlled slow-vs-fast; no speed condition labels.', 'recommendation': 'Replace with natural movement validation; optionally report observed speed distribution.'},
    {'paper_claim_or_factor': 'With and without finger contact', 'supported_by_current_data': 'no for comparison', 'evidence': 'User states only without finger / without contact is available for this analysis.', 'recommendation': 'Remove contact-vs-no-contact comparison, or collect with-contact trials.'},
    {'paper_claim_or_factor': 'MediaPipe spatial accuracy vs simultaneous OptiTrack', 'supported_by_current_data': 'yes', 'evidence': '8 synchronized matched segments after +48.5 s MP->OT time shift; metrics computed.', 'recommendation': 'Keep, but state calibrated 2D X-Z accuracy and include synchronization/calibration details.'},
]
coverage = pd.DataFrame(coverage_rows)
coverage.to_csv(OUT_DIR / 'paper_claim_scope_audit.csv', index=False)
print(coverage.to_string(index=False))

In [ ]:
errs = labeled['cv_error_mm'].dropna().to_numpy(dtype=float) if 'cv_error_mm' in labeled.columns else np.array([])
if len(errs):
    overall = {'rmse': float(np.sqrt(np.mean(errs**2))), 'mean': float(np.mean(errs)), 'p95': float(np.percentile(errs, 95)), 'n': int(len(errs))}
else:
    overall = {
        'rmse': float(np.sqrt(np.average(segment_metrics['rmse_mm']**2, weights=segment_metrics['samples_compared']))),
        'mean': float(np.average(segment_metrics['mean_positional_error_mm'], weights=segment_metrics['samples_compared'])),
        'p95': float(np.average(segment_metrics['p95_positional_error_mm'], weights=segment_metrics['samples_compared'])),
        'n': int(segment_metrics['samples_compared'].sum()),
    }

revised = f"""# Recommended conservative paper wording

## My recommendation

Do **not** include the untested factors as completed experiments. It is not a scientific problem that they were not tested, but it is a reporting problem if the paper claims them.

Specifically remove or rewrite:

- three hand orientations (N, NE, NW)
- clockwise/counterclockwise comparison
- five slow/five fast repetitions
- with-vs-without finger contact comparison

unless you collect those data.

## Suggested replacement text

The full system was validated under closed-loop operation using simultaneous OptiTrack and overhead camera tracking. Targets were arranged at radial distances of 0, 1, 2, 3, 5, and 8 cm along eight cardinal and diagonal directions for four fingers. Each trial followed a natural inward-outward radial trajectory between the outer workspace and the center. OptiTrack operated at 120 Hz, and the overhead camera recording was used for MediaPipe-based 2D hand tracking. Corresponding target-grid locations were used to align the MediaPipe image coordinates to the OptiTrack X-Z plane using an affine calibration, with the workspace center used as the common origin. The current validation was performed without finger contact with the tactor workspace.

Across the matched synchronized recordings, MediaPipe spatial accuracy relative to OptiTrack showed a calibrated 2D mean positional error of {overall['mean']:.2f} mm, RMSE of {overall['rmse']:.2f} mm, and 95th-percentile positional error of {overall['p95']:.2f} mm (n={overall['n']} samples). Per-finger and per-target-location errors are reported in the supplementary analysis tables.

## If you want to keep the stronger original claims

Then collect additional data with explicit condition labels:

1. hand orientation: N, NE, NW
2. traversal order: clockwise and counterclockwise
3. movement speed: slow and fast, with controlled timing or speed labels
4. contact condition: with and without finger/tactor contact

Without those data, I would not claim these factors in the paper.
"""

(OUT_DIR / 'recommended_conservative_paper_text.md').write_text(revised, encoding='utf-8')
print(revised)